# Lab 12: Orthogonality — Clean Directions and Stable Coordinates

This lab accompanies Chapter 12. The goal is not only to compute dot products, but to understand why orthogonality is one of the most important ideas in applied linear algebra.

We will study:

- orthogonal and orthonormal vectors;
- the Pythagorean theorem in vector spaces;
- orthogonal projection onto lines and subspaces;
- Gram--Schmidt as repeated explanation removal;
- QR factorization;
- least squares with QR;
- orthogonality in signals and high-dimensional data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

## Part 1. Dot product as overlap

The dot product measures alignment. When the dot product is zero, the vectors have no overlap in each other's direction.

In [ ]:
u = np.array([2, 1])
v = np.array([-1, 2])

print("u dot v =", u @ v)
print("||u|| =", np.linalg.norm(u))
print("||v|| =", np.linalg.norm(v))

plt.figure(figsize=(6,6))
plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.quiver(0,0,u[0],u[1],angles='xy',scale_units='xy',scale=1,label='u')
plt.quiver(0,0,v[0],v[1],angles='xy',scale_units='xy',scale=1,label='v')
plt.xlim(-3,3); plt.ylim(-3,3); plt.gca().set_aspect('equal')
plt.grid(True); plt.legend(); plt.title('Orthogonal vectors')
plt.show()

### Student task

Change the vector `v`. Find three different vectors orthogonal to `u = [2, 1]`. What pattern do you see?

## Part 2. Orthogonal sets and Gram matrices

A fast way to check whether many vectors are orthonormal is to place them as columns of a matrix $Q$ and compute $Q^TQ$.

In [ ]:
Q = np.array([[1/np.sqrt(2),  1/np.sqrt(2)],
              [1/np.sqrt(2), -1/np.sqrt(2)]])
print("Q =")
print(Q)
print("Q^T Q =")
print(Q.T @ Q)

In [ ]:
# A 3D orthonormal set
Q3 = np.eye(3)
print(Q3.T @ Q3)

# A non-orthonormal set
A = np.array([[1, 1],
              [0, 1],
              [1, 0]], dtype=float)
print("A^T A =")
print(A.T @ A)

## Part 3. Pythagorean theorem in vector spaces

When vectors are orthogonal, squared lengths add. This is one reason orthogonality is so useful in error analysis.

In [ ]:
u = np.array([2,1])
v = np.array([-1,2])

left = np.linalg.norm(u+v)**2
right = np.linalg.norm(u)**2 + np.linalg.norm(v)**2

print("||u+v||^2 =", left)
print("||u||^2 + ||v||^2 =", right)
print("difference =", left-right)

## Part 4. Projection onto an orthonormal subspace

If $Q$ has orthonormal columns, projection is easy:

$$
\hat{x}=QQ^Tx.
$$

In [ ]:
q1 = np.array([1,1,0], dtype=float) / np.sqrt(2)
q2 = np.array([0,0,1], dtype=float)
Q = np.column_stack([q1,q2])
x = np.array([3,1,4], dtype=float)

x_hat = Q @ (Q.T @ x)
r = x - x_hat

print("Q^T Q =")
print(Q.T @ Q)
print("projection =", x_hat)
print("residual =", r)
print("Q^T residual =", Q.T @ r)

In [ ]:
fig = plt.figure(figsize=(7,6))
ax = fig.add_subplot(111, projection='3d')
ax.quiver(0,0,0,x[0],x[1],x[2],length=1,normalize=False,label='x')
ax.quiver(0,0,0,x_hat[0],x_hat[1],x_hat[2],length=1,normalize=False,label='projection')
ax.quiver(x_hat[0],x_hat[1],x_hat[2],r[0],r[1],r[2],length=1,normalize=False,label='residual')
ax.set_xlim(0,4); ax.set_ylim(-1,3); ax.set_zlim(0,5)
ax.set_title('Projection plus orthogonal residual')
ax.legend()
plt.show()

## Part 5. Gram--Schmidt from scratch

Gram--Schmidt repeatedly removes the part of a vector already explained by earlier directions.

In [ ]:
def classical_gram_schmidt(A):
    A = A.astype(float)
    m, n = A.shape
    Q = np.zeros((m,n))
    R = np.zeros((n,n))
    for j in range(n):
        v = A[:,j].copy()
        for i in range(j):
            R[i,j] = Q[:,i] @ A[:,j]
            v = v - R[i,j] * Q[:,i]
        R[j,j] = np.linalg.norm(v)
        if R[j,j] < 1e-12:
            raise ValueError("Columns are linearly dependent or nearly dependent.")
        Q[:,j] = v / R[j,j]
    return Q, R

A = np.array([[1,1],
              [1,2],
              [0,1]], dtype=float)
Q, R = classical_gram_schmidt(A)
print("Q =")
print(Q)
print("R =")
print(R)
print("Q^T Q =")
print(Q.T @ Q)
print("A - QR norm =", np.linalg.norm(A - Q @ R))

### Student task

Modify `A` so that its columns are nearly parallel. What happens to the Gram--Schmidt process? Compare with `np.linalg.qr(A)`.

## Part 6. QR factorization and least squares

QR gives a stable way to solve least-squares problems.

In [ ]:
# Fit y = beta0 + beta1 x to noisy data
rng = np.random.default_rng(12)
t = np.linspace(0, 10, 30)
y = 2.0 + 0.7*t + rng.normal(0, 0.8, size=t.shape)

A = np.column_stack([np.ones_like(t), t])
Q, R = np.linalg.qr(A)
beta_qr = np.linalg.solve(R, Q.T @ y)
beta_normal = np.linalg.solve(A.T @ A, A.T @ y)

print("QR solution:", beta_qr)
print("normal-equation solution:", beta_normal)

plt.figure(figsize=(7,5))
plt.scatter(t, y, label='data')
plt.plot(t, A @ beta_qr, label='least-squares line')
plt.title('Least squares via QR')
plt.xlabel('t'); plt.ylabel('y'); plt.legend(); plt.grid(True)
plt.show()

## Part 7. Signals as orthogonal patterns

Orthogonal patterns allow a signal to be decomposed into independent ingredients. Here we use simple sine and cosine waves.

In [ ]:
n = 200
xgrid = np.linspace(0, 2*np.pi, n, endpoint=False)
patterns = np.column_stack([
    np.sin(xgrid),
    np.cos(xgrid),
    np.sin(2*xgrid),
    np.cos(2*xgrid)
])
# Normalize columns
Q = patterns / np.linalg.norm(patterns, axis=0)
print("Q^T Q approximately =")
print(Q.T @ Q)

coeff = np.array([2.5, -1.0, 0.8, 1.6])
signal = Q @ coeff
recovered = Q.T @ signal
print("original coefficients:", coeff)
print("recovered coefficients:", recovered)

plt.figure(figsize=(8,4))
plt.plot(xgrid, signal)
plt.title('A signal built from orthonormal wave patterns')
plt.grid(True)
plt.show()

## Part 8. High-dimensional random orthogonality

In high dimensions, random directions are often nearly orthogonal. This fact is important in machine learning, random projections, embeddings, and high-dimensional statistics.

In [ ]:
rng = np.random.default_rng(0)
dims = [2, 5, 10, 50, 100, 500]
num_pairs = 3000
means = []
stds = []

for d in dims:
    X = rng.normal(size=(num_pairs, d))
    Y = rng.normal(size=(num_pairs, d))
    X = X / np.linalg.norm(X, axis=1, keepdims=True)
    Y = Y / np.linalg.norm(Y, axis=1, keepdims=True)
    cosines = np.sum(X*Y, axis=1)
    means.append(np.mean(np.abs(cosines)))
    stds.append(np.std(cosines))

plt.figure(figsize=(7,5))
plt.plot(dims, means, marker='o', label='mean |cosine|')
plt.plot(dims, stds, marker='s', label='std cosine')
plt.xscale('log')
plt.xlabel('dimension')
plt.ylabel('value')
plt.title('Random directions become nearly orthogonal')
plt.grid(True)
plt.legend()
plt.show()

## Part 9. Reflection questions

1. Why is $Q^TQ=I$ such a powerful condition?
2. Why does projection become simpler with an orthonormal basis?
3. In Gram--Schmidt, what does the residual represent?
4. Why can QR be more stable than normal equations?
5. Why is near-orthogonality common in high-dimensional random data?